# Laboratório 2 — O número que muda a decisão

**Aprendizagem de Máquina Supervisionada — 2026.2 · Aula 2 (10/09)**

Hoje **não treinamos nenhum modelo.** O escore de risco é dado como insumo, como
se viesse de um sistema acadêmico já em operação. Na simulação, ele representa uma
probabilidade calibrada. O objetivo vem antes do ajuste de modelos:
mostrar que **a mesma previsão produz decisões diferentes** conforme a capacidade
de atendimento e o custo dos erros.

A partir de 15/10 aprenderemos a estimar esse escore. Antes disso, precisamos
saber o que fazer com ele.

**Ao final você será capaz de:** calcular prevalência e o baseline trivial;
comparar regras de ordenação por `precision@k`; ler uma curva de ganho; e derivar
e verificar numericamente o limiar ótimo sob custos assimétricos.

**Base conceitual:** Morettin e Singer, cap. 1, seções 1.5–1.6. **Apoio em
Python e avaliação preditiva:** ISLP, caps. 2 e 4.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

SEED = 20260910
rng = np.random.default_rng(SEED)

# Paleta categórica validada (ordem fixa; nunca reciclar)
COR = {"aleatoria": "#8E5AA8", "faltas": "#C4791A", "escore": "#2D6FB8"}
TINTA = "#253342"
TINTA_FRACA = "#657484"

pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. A base

**Dados simulados**, gerados com semente fixa. Não representam nenhuma instituição
real e não devem ser usados como evidência empírica de coisa alguma. Servem apenas
para que a aritmética da decisão fique visível.

Unidade de análise: **estudante–semana**. Alvo: abandono nos **próximos 60 dias**.
Corte: domingo. Todas as variáveis abaixo são observáveis **antes** do corte.

In [2]:
n = 4000

faltas_acum = rng.poisson(3.0, n)
media_notas = np.clip(rng.normal(6.5, 1.5, n), 0, 10)
reprovacoes = rng.poisson(0.8, n)
dias_sem_acesso = rng.exponential(7.0, n)

# Processo gerador do rótulo (desconhecido na vida real)
eta = (
    -3.92
    + 0.18 * faltas_acum
    + 0.35 * reprovacoes
    + 0.06 * dias_sem_acesso
    - 0.25 * (media_notas - 6.5)
)
p_verdadeiro = 1 / (1 + np.exp(-eta))
abandono = rng.binomial(1, p_verdadeiro)

# Escore fornecido para a aula: probabilidade calibrada do processo simulado.
# Ainda há erro irredutível, pois o desfecho é uma realização de Bernoulli.
escore_sistema = p_verdadeiro.copy()

df = pd.DataFrame(
    {
        "faltas_acum": faltas_acum,
        "media_notas": media_notas,
        "reprovacoes": reprovacoes,
        "dias_sem_acesso": dias_sem_acesso,
        "escore_sistema": escore_sistema,
        "abandono": abandono,
    }
)
print(f"linhas: {len(df):,}")
df.head()

linhas: 4,000


,faltas_acum,media_notas,reprovacoes,dias_sem_acesso,escore_sistema,abandono
0,0,3.5741,1,4.8269,0.0725,0
1,3,4.6674,0,5.1417,0.0683,0
2,4,8.0711,0,8.1786,0.0430,0
3,4,7.0737,0,1.4777,0.0372,0
4,3,4.0783,1,1.1368,0.0866,0


## 2. Prevalência e o baseline que engana

Antes de qualquer métrica sofisticada: **quantos casos positivos existem?**

In [3]:
prevalencia = df["abandono"].mean()
acuracia_sempre_negativo = 1 - prevalencia

print(f"prevalência de abandono ........... {prevalencia:.1%}")
print(f"acurácia de 'sempre negativo' ..... {acuracia_sempre_negativo:.1%}")
print(f"positivos encontrados por ela ..... {0} de {int(df['abandono'].sum())}")

prevalência de abandono ........... 8.1%
acurácia de 'sempre negativo' ..... 91.9%
positivos encontrados por ela ..... 0 de 325


> **Registre:** um sistema com acurácia de aproximadamente 92% que nunca encontra
> um único caso. Acurácia alta em base desbalanceada não é evidência de utilidade.
> A pergunta certa não é "acerta quanto?", e sim **"quem entra na fila de contato?"**

## 3. A restrição de capacidade

A coordenação consegue contatar **50 estudantes por semana**. Isso muda o problema:
não interessa acertar em toda a população, interessa **ordenar bem os 50 primeiros**.

Comparamos três regras de ordenação. Nenhuma delas é um modelo treinado.

In [4]:
K = 50


def avalia_ordenacao(nome, ordem, y, k=K):
    """Métricas de uma regra de ordenação sob capacidade k."""
    topo = ordem[:k]
    capturados = int(y[topo].sum())
    total_pos = int(y.sum())
    prec = capturados / k
    rec = capturados / total_pos
    return {
        "regra": nome,
        f"precision@{k}": prec,
        f"recall@{k}": rec,
        "capturados": capturados,
        "lift": prec / y.mean(),
    }


y = df["abandono"].to_numpy()

ordem_aleatoria = rng.permutation(len(df))
ordem_faltas = np.argsort(-df["faltas_acum"].to_numpy(), kind="stable")
ordem_escore = np.argsort(-df["escore_sistema"].to_numpy(), kind="stable")

# Uma única extração aleatória é ruidosa com k = 50. Repetimos 200 vezes e
# tomamos a média — o baseline aleatório é uma expectativa, não um sorteio.
repeticoes = [
    avalia_ordenacao("Seleção aleatória", rng.permutation(len(df)), y)
    for _ in range(200)
]
media_aleatoria = {
    "regra": "Seleção aleatória (média de 200)",
    f"precision@{K}": float(np.mean([r[f"precision@{K}"] for r in repeticoes])),
    f"recall@{K}": float(np.mean([r[f"recall@{K}"] for r in repeticoes])),
    "capturados": float(np.mean([r["capturados"] for r in repeticoes])),
    "lift": float(np.mean([r["lift"] for r in repeticoes])),
}

comparacao = pd.DataFrame(
    [
        media_aleatoria,
        avalia_ordenacao("Regra atual: mais faltas", ordem_faltas, y),
        avalia_ordenacao("Escore do sistema", ordem_escore, y),
    ]
)
comparacao

,regra,precision@50,recall@50,capturados,lift
0,Seleção aleatória (média de 200),0.0822,0.0126,4.1100,1.0117
1,Regra atual: mais faltas,0.2600,0.0400,13.0000,3.2000
2,Escore do sistema,0.4400,0.0677,22.0000,5.4154


**Leia a coluna `lift`.** Ela responde à pergunta do gestor: *quantas vezes melhor
que sortear?* É essa razão, e não a acurácia, que justifica o custo de manter o
sistema.

> **Pare e responda antes de seguir:** a regra atual (mais faltas) já é melhor que
> o acaso. Se o escore do sistema custasse R\$ 20 mil por ano para ser mantido, o
> ganho sobre a regra atual pagaria essa conta? Que informação falta para responder?


- O Custo de Oportunidade Zero (Seleção Aleatória): Escolhendo 50 alunos às cegas, a coordenação acertaria o alvo em média 4 vezes (capturados = 4,11). A precision@50 de 8,2% reflete basicamente a taxa de prevalência de abandono na base inteira. O lift próximo de 1 confirma: agir aleatoriamente não traz ganho de eficiência.

- O Baseline Operacional (Regra de Mais Faltas): Usando a heurística que provavelmente já existe hoje na coordenação (ordenar por faltas acumuladas), a eficiência melhora. Dos 50 contatos, 13 seriam alunos em risco real de abandono. A precisão salta para 26% e a estratégia se mostra 3,2 vezes melhor que o acaso (lift = 3.2).

- O Modelo Preditivo (Escore do Sistema): Guiando as mesmas 50 ligações pelo escore do modelo, a coordenação intercepta 22 estudantes prestes a abandonar. A precisão sobe para 44%, tornando a ação 5,4 vezes mais eficiente do que tentar a sorte (lift = 5.41).

## 4. Curva de ganho

A tabela acima olha apenas um ponto ($k = 50$). A curva de ganho mostra o
desempenho para **toda** capacidade possível: no eixo horizontal, quantos
estudantes são contatados; no vertical, quantos casos de abandono são alcançados.

In [5]:
def curva_ganho(ordem, y):
    return np.concatenate([[0], np.cumsum(y[ordem])])


x = np.arange(len(df) + 1)
total_pos = int(y.sum())

fig = go.Figure()
for nome, ordem, chave in [
    ("Escore do sistema", ordem_escore, "escore"),
    ("Regra atual: mais faltas", ordem_faltas, "faltas"),
    ("Seleção aleatória", ordem_aleatoria, "aleatoria"),
]:
    fig.add_trace(
        go.Scatter(
            x=x,
            y=curva_ganho(ordem, y),
            name=nome,
            mode="lines",
            line=dict(color=COR[chave], width=2),
            hovertemplate="contatados: %{x}<br>abandonos alcançados: %{y}<extra>" + nome + "</extra>",
        )
    )

fig.add_vline(x=K, line_width=1, line_dash="dot", line_color=TINTA_FRACA)
fig.add_annotation(x=K, y=total_pos * 0.97, text=f"capacidade = {K}", showarrow=False,
                   xanchor="left", xshift=6, font=dict(size=11, color=TINTA_FRACA))
# O gráfico é interativo: aproxime o eixo x até ~200 para ver o que ocorre na
# altura da capacidade. É lá que a decisão acontece.
fig.update_layout(
    title="Quantos abandonos cada regra alcança, por número de contatos",
    xaxis_title="estudantes contatados (ordenados pela regra)",
    yaxis_title="abandonos alcançados",
    template="plotly_white",
    font=dict(color=TINTA),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(t=90, r=20),
    height=420,
)
fig.show()

> **Aproxime o eixo horizontal até 200** — o gráfico é interativo. A distância
> vertical entre as curvas, na altura da capacidade, **é** o valor do
> sistema. Se as curvas se tocam em $k$, o sistema não está entregando nada ali —
> ainda que a acurácia seja excelente.

## 5. O limiar ótimo sob custos assimétricos

Agora suponha que **não haja restrição de capacidade**: a organização age sempre
que o risco justificar. Quando vale a pena agir?

Sejam:

- $B$ — o **benefício líquido** de agir sobre um estudante que de fato abandonaria;
- $C$ — o **custo** de agir sobre um estudante que não abandonaria.

Agir sobre um caso com risco estimado $\hat p$ tem valor esperado

$$\mathbb{E}[\text{agir}] = \hat p \cdot B - (1-\hat p)\cdot C .$$

Não agir tem valor $0$. Logo, agir compensa quando

$$\hat p \cdot B - (1-\hat p)\cdot C > 0
\quad\Longleftrightarrow\quad
\hat p > \frac{C}{B+C} \equiv t^{*} .$$

**O limiar não é 0,5.** Ele sai da razão entre os custos dos dois erros — e é uma
decisão da organização, não do algoritmo.

In [6]:
B = 10.0   # benefício de reter um estudante que abandonaria
C = 1.0    # custo de contatar quem não abandonaria

t_teorico = C / (B + C)
print(f"limiar ótimo teórico  t* = C/(B+C) = {C:.0f}/({B:.0f}+{C:.0f}) = {t_teorico:.4f}")
print(f"ou seja: vale contatar quem tem risco acima de {t_teorico:.1%}")

limiar ótimo teórico  t* = C/(B+C) = 1/(10+1) = 0.0909
ou seja: vale contatar quem tem risco acima de 9.1%


Verificamos empiricamente: varremos o limiar e calculamos o valor total da política.

In [7]:
def valor_da_politica(escore, y, t, b=B, c=C):
    agir = escore > t
    return float((y[agir] * b - (1 - y[agir]) * c).sum())


grade = np.linspace(0.01, 0.60, 300)
valores = np.array([valor_da_politica(df["escore_sistema"].to_numpy(), y, t) for t in grade])
t_empirico = grade[int(np.argmax(valores))]

print(f"limiar empírico ótimo ......... {t_empirico:.4f}")
print(f"limiar teórico  t* ............ {t_teorico:.4f}")
print(f"valor no ótimo empírico ....... {valores.max():,.1f}")
print(f"valor em t = 0,50 ............. {valor_da_politica(df['escore_sistema'].to_numpy(), y, 0.50):,.1f}")

limiar empírico ótimo ......... 0.0929
limiar teórico  t* ............ 0.0909
valor no ótimo empírico ....... 757.0
valor em t = 0,50 ............. 17.0


In [8]:
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=grade, y=valores, mode="lines",
        line=dict(color=COR["escore"], width=2),
        hovertemplate="limiar: %{x:.3f}<br>valor da política: %{y:,.0f}<extra></extra>",
        showlegend=False,
    )
)
fig2.add_vline(x=t_teorico, line_width=2, line_dash="dash", line_color=COR["faltas"])
fig2.add_annotation(x=t_teorico, y=valores.max(), text=f"t* = {t_teorico:.3f}",
                    showarrow=False, xanchor="left", xshift=8, yshift=22,
                    font=dict(size=12, color=COR["faltas"]))
fig2.update_layout(
    title="Valor esperado da política em função do limiar de ação",
    xaxis_title="limiar aplicado ao escore de risco",
    yaxis_title="valor total da política",
    template="plotly_white",
    font=dict(color=TINTA),
    margin=dict(t=70, r=20),
    height=400,
)
fig2.show()

> Como o escore desta simulação é uma probabilidade calibrada, o ótimo empírico
> tende a ficar próximo de $t^{*}$; a diferença decorre da realização aleatória
> dos desfechos e do tamanho finito da amostra. Em dados reais, o limiar só tem
> essa interpretação direta quando a probabilidade está bem calibrada. Calibração
> será retomada em 19/11.

## 6. Quando as duas coisas colidem

O limiar ótimo diz *quem vale a pena contatar*. A capacidade diz *quantos cabem*.
Nem sempre são compatíveis.

In [9]:
escore = df["escore_sistema"].to_numpy()
n_acima_do_limiar = int((escore > t_teorico).sum())
limiar_efetivo = float(np.sort(escore)[-K])

print(f"estudantes acima de t* = {t_teorico:.3f} ........ {n_acima_do_limiar:,}")
print(f"capacidade semanal ......................... {K}")
print(f"limiar efetivo imposto pela capacidade ..... {limiar_efetivo:.4f}")
print()
print(f"valor da política sob t* (sem restrição) ... {valor_da_politica(escore, y, t_teorico):,.1f}")
print(f"valor da política sob capacidade k={K} ...... {float((y[ordem_escore[:K]] * B - (1 - y[ordem_escore[:K]]) * C).sum()):,.1f}")

estudantes acima de t* = 0.091 ........ 1,094
capacidade semanal ......................... 50
limiar efetivo imposto pela capacidade ..... 0.2986

valor da política sob t* (sem restrição) ... 721.0
valor da política sob capacidade k=50 ...... 192.0


> **A escassez é a restrição que morde.** O limiar economicamente ótimo selecionaria
> muito mais gente do que a equipe consegue atender, então o limiar que vale de fato
> é o que a capacidade impõe. É por isso que, num projeto com fila, a métrica que
> orienta a decisão é `precision@k`. Acurácia e AUC podem continuar como
> diagnósticos, mas não medem diretamente o resultado dentro dos 50 atendimentos.

## 7. Exercício para levar ao Canvas

Com os dados **do seu projeto** (ou com as hipóteses da escuta, se os dados ainda
não chegaram):

1. Qual é a capacidade $k$ do usuário, por período de decisão?
2. Qual é o baseline operacional — a regra que ordenaria a fila hoje, sem modelo?
3. Estime $B$ e $C$ em unidade comum (reais, horas de equipe, atendimentos).
   Não precisa ser exato; precisa ser **defensável**.
4. Calcule $t^{*} = C/(B+C)$ e escreva o que ele significa em uma frase de gestão.
5. Registre qual das duas restrições morde primeiro no seu caso: o limiar ou a fila.

**Entrega:** essas cinco respostas entram na Task 2A e alimentam o Canvas de 17/09.

### Transferência para outros setores

Escolha um caso e reescreva as cinco respostas: uma empresa que oferece retenção
a 100 clientes por semana; uma secretaria que prioriza 30 estabelecimentos para
fiscalização; ou uma unidade de saúde que telefona para 40 pacientes com risco de
faltar à consulta. Em cada caso, identifique quem decide e qual consequência um
falso positivo produz.